# Livrable 1 : Modélisation du Problème d'Optimisation de Tournée (CesiCDP)

## 1. Introduction et Contexte (Storytelling)

Depuis les années 90, la prise de conscience mondiale face aux enjeux climatiques s'est matérialisée par divers accords (Kyoto, Paris) visant à réduire drastiquement les émissions de gaz à effet de serre. Dans ce contexte, l'ADEME (Agence de l'Environnement et de la Maîtrise de l'Énergie) a lancé un appel à projets pour concevoir de nouvelles solutions de *Mobilité Multimodale Intelligente*.

L'équipe **CesiCDP**, experte en optimisation et logistique, se positionne sur cet appel. Notre objectif est de concevoir un système algorithmique capable de réduire les temps de parcours, les coûts, et *in fine* l'empreinte carbone liée à la livraison de marchandises. 

Dans de nombreux cas réels, les livreurs ne peuvent pas passer à n'importe quelle heure (un client n'est présent qu'en matinée, un magasin n'accepte les livraisons qu'entre 8h et 10h). De plus, le réseau routier n'est pas infaillible : des travaux, des accidents ou des zones à faibles émissions peuvent restreindre ou renchérir l'utilisation de certaines routes.

Pour répondre à ces problématiques, nous modélisons un problème d'optimisation de tournée de véhicule s'apparentant au **Traveling Salesperson Problem (TSP)**, mais enrichi de deux contraintes majeures :
1. **Fenêtres Temporelles (Time Windows)** : Chaque point de livraison possède un intervalle de temps strict où le passage est autorisé.
2. **Coût ou restriction de passage sur certaines arêtes** : Les arêtes du graphe ont des coûts variables et certaines peuvent être totalement infranchissables.

## 2. Modélisation Formelle et Mathématique

Nous modélisons notre réseau sous la forme d'un graphe orienté $G = (V, E)$, où :
* $V = \{0, 1, 2, ..., n\}$ est l'ensemble des nœuds. Le nœud $0$ représente le point de départ et d'arrivée (le dépôt), et les nœuds $1$ à $n$ représentent les villes/clients à visiter.
* $E \subseteq V \times V$ est l'ensemble des arêtes (routes) reliant ces villes.

### 2.1. Données et Paramètres
* $c_{ij}$ : Le coût associé au parcours de l'arête $(i, j) \in E$. **Contrainte de restriction de passage :** Certaines routes peuvent être plus coûteuses. Si une route est totalement interdite (par exemple, **travaux ou routes bloquées**), on pose mathématiquement $c_{ij} = +\infty$ pour empêcher l'algorithme de l'emprunter.
* $d_{ij}$ : La durée de trajet (temps de transport) entre le nœud $i$ et le nœud $j$.
* $[e_i, l_i]$ : La **fenêtre temporelle (Time Window)** associée au nœud $i$. Par exemple, si une ville est **disponible uniquement de 8 h à 10 h**, alors $e_i = 8$ et $l_i = 10$.
* $M$ : Une constante analytique très grande (Big-M method) utilisée pour linéariser les contraintes conditionnelles.

### 2.2. Variables de Décision
Nous avons besoin de deux types de variables pour résoudre ce problème :
1. **Variables de routage (binaires) :** 
   $x_{ij} \in \{0, 1\}$ qui vaut $1$ si le véhicule emprunte l'arête $(i, j)$, et $0$ sinon.
2. **Variables temporelles (continues) :** 
   $t_i \ge 0$ qui correspond à l'heure à laquelle le véhicule commence son service au nœud $i$.

### 2.3. Fonction Objectif
L'objectif est de minimiser le coût total parcouru par le véhicule (ou le temps total).
$$ \min \sum_{i \in V} \sum_{j \in V} c_{ij} x_{ij} $$

### 2.4. Contraintes

**1. Contraintes de degré (Visite unique) :**
Chaque ville doit être visitée exactement une fois par le véhicule.
* Un seul arc sort de chaque ville $i$ :
$$ \sum_{j \in V, j \neq i} x_{ij} = 1 \quad \forall i \in V \setminus \{0\} $$
* Un seul arc entre dans chaque ville $j$ :
$$ \sum_{i \in V, i \neq j} x_{ij} = 1 \quad \forall j \in V \setminus \{0\} $$

**2. Départ et Retour au dépôt (Nœud 0) :**
Le véhicule doit quitter le dépôt et y revenir.
$$ \sum_{j \in V, j \neq 0} x_{0j} = 1 $$
$$ \sum_{i \in V, i \neq 0} x_{i0} = 1 $$

**3. Continuité temporelle (et élimination des sous-tours) :**
Si le véhicule se rend du nœud $i$ au nœud $j$ ($x_{ij} = 1$), l'heure de début de service à $j$ ($t_j$) doit être supérieure ou égale à l'heure de début de service à $i$ ($t_i$) plus le temps de trajet $d_{ij}$. La constante $M$ désactive la contrainte si la route n'est pas empruntée ($x_{ij} = 0$).
$$ t_i + d_{ij} - M(1 - x_{ij}) \le t_j \quad \forall i, j \in V, i \neq j $$

**4. Respect strict des Fenêtres Temporelles (Time Windows) :**
Chaque ville doit être visitée dans un certain intervalle de temps. Le parcours doit respecter la plage de disponibilité de chaque ville (par exemple de 8h à 10h) afin d'autoriser le service sur place à l'heure dite $t_i$.
$$ e_i \le t_i \le l_i \quad \forall i \in V $$

**5. Respect des restrictions de passage (Routes coûteuses ou bloquées) :**
Le domaine des arrêtes permet déjà des coûts supplémentaires grâce au paramètre $c_{ij}$. Pour garantir l'impossibilité pure et simple du passage en cas de travaux, on force la variable binaire à zéro pour toute arête $(i, j)$ dont le coût a été défini comme infini (route barrée).
$$ x_{ij} = 0 \quad \text{si l'accès est bloqué } (c_{ij} = +\infty) $$

### 2.5. Représentation Graphique du Problème

Pour rendre notre modélisation plus concrète et illustrer la démarche pour l'ADEME, voici une modélisation visuelle simplifiée d'une de nos futures instances de test. Nous utilisons la librairie `networkx` pour afficher notre graphe où figurent :
* Les **fenêtres de temps** affichées sous les nœuds.
* Les **coûts** normaux des routes en noir.
* Les **routes bloquées (travaux, interdiction)** avec un coût affiché comme infini ($\infty$) en rouge pointillé.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# 1. Création d'un graphe orienté
G = nx.DiGraph()

# 2. Ajout des villes (nœuds) avec attributs "Label" pour afficher le nom et la Time Window
nodes = {
    0: {"pos": (0, 0), "label": "Dépôt\n[0h, 24h]", "color": "lightgreen"},
    1: {"pos": (2, 2), "label": "Ville 1\n[8h, 10h]", "color": "lightblue"},
    2: {"pos": (5, 1), "label": "Ville 2\n[10h, 12h]", "color": "lightblue"},
    3: {"pos": (2, -2), "label": "Ville 3\n[14h, 16h]", "color": "lightgreen"},
}

for n, attr in nodes.items():
    G.add_node(n, pos=attr["pos"], label=attr["label"], color=attr["color"])

# 3. Ajout des routes normales avec leur coût/temps de trajet
normal_edges = [
    (0, 1, 30), (1, 2, 45), (2, 3, 20),
    (3, 0, 50), (0, 3, 40), (0, 2, 60)
]
for u, v, w in normal_edges:
    G.add_edge(u, v, weight=w)

# 4. Ajout d'une route bloquée (travaux - coût infini)
blocked_edge = (1, 3)
G.add_edge(blocked_edge[0], blocked_edge[1], weight="∞")

# Configuration de l'affichage matplotlib
plt.figure(figsize=(10, 6))
pos = nx.get_node_attributes(G, 'pos')
node_colors = [data["color"] for _, data in G.nodes(data=True)]

# Dessin des nœuds avec leur Time Window
nx.draw_networkx_nodes(G, pos, node_size=3500, node_color=node_colors, edgecolors="gray")
nx.draw_networkx_labels(G, pos, labels=nx.get_node_attributes(G, 'label'), font_size=10, font_weight="bold")

# Dessin des arêtes normales (flèches grises)
nx.draw_networkx_edges(G, pos, edgelist=normal_edges, arrowstyle='->', arrowsize=20, edge_color='#555555', connectionstyle='arc3,rad=0.1')

# Dessin de la route barrée (flèche rouge pointillée)
nx.draw_networkx_edges(G, pos, edgelist=[blocked_edge], style='dashed', edge_color='red', arrowstyle='-|>')

# Affichage des coûts près des flèches
edge_labels = {(u, v): d['weight'] for u, v, d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, label_pos=0.3, font_color="darkblue", font_size=11)

plt.title("Réseau de Logistique: Fenêtres de Temps et Routes Bloquées", fontsize=14, fontweight="bold")
plt.axis("off")
plt.tight_layout()
plt.show()

## 3. Analyse de la Complexité Théorique

L'élaboration d'un algorithme pour résoudre un problème mathématique nécessite au préalable son analyse pour évaluer la difficulté de sa résolution en fonction de la taille de l'instance d'entrée ($n$ villes).

### 3.1. Complexité du TSP de base
Le Problème du Voyageur de Commerce (TSP - Traveling Salesperson Problem) standard consiste à trouver le plus petit cycle hamiltonien complet dans un graphe pondéré. 
Le TSP classique est un problème de décision connu depuis longtemps comme appartenant à la classe **NP-Complet** (démontré formellement par Richard Karp en 1972). Le problème d'optimisation (rechercher le minimum) est donc **NP-Difficile**.
L'espace des solutions, pour un graphe complet de $n$ sommets repassant par le dépôt, comprend $(n-1)!$ tournées possibles. Une approche par force brute (ex: générer toutes les combinaisons) a une complexité en temps de $\mathcal{O}(n!)$, ce qui la rend inexploitable en pratique même au-delà d'une quinzaine de nœuds.

### 3.2. Impact des Contraintes Ajoutées (TSPTW)
Notre modèle n'est pas qu'un simple TSP, il s'agit d'un **TSP with Time Windows (TSPTW)** assujetti à certaines suppressions de graphe.

**Est-ce plus difficile ?**
Le TSPTW est une généralisation du TSP. Si l'on choisit de régler $e_i = 0$ et $l_i = +\infty$ pour chaque nœud (des fenêtres temporelles infiniment larges) et que toutes les arêtes sont praticables, notre TSPTW retombe très exactement sur le TSP classique. 
Puisque le problème est au moins aussi difficile que le TSP classique, il est par réduction **NP-Difficile** au sens strict.

Cependant, *en pratique*, pour certains algorithmes (comme le Brand & Bound ou la programmation dynamique), l'ajout de fenêtres temporelles induit une restriction forte de l'espace des solutions (le nombre de parcours valides qui respectent les fenêtres est très inférieur à $n!$). Ainsi, des instances assez larges peuvent parfois être résolues plus vite car l'algorithme "coupe" de nombreuses branches non valides de son arbre d'exploration temporelle. 

Mais dans le pire des cas, trouver *ne serait-ce qu'une solution valide/réalisable* pour le TSPTW (sans même parler d'optimisation) est un problème **NP-Complet** (Savelsbergh, 1985).

*En conclusion :*
L'impossibilité de résoudre de larges instances de manière exacte en un temps polynomial justifie pleinement, pour la suite de ce projet, de développer à la fois des algorithmes exacts pour les petites instances (Validation) et des approches approchées ou méta-heuristiques (Recuit simulé, Algorithme génétique) pour les territoires logistiques comportant de nombreux nœuds géographiques.

## 4. Bibliographie / Références Scientifiques

1. **Garey, M. R., & Johnson, D. S. (1979).** *Computers and Intractability: A Guide to the Theory of NP-Completeness*. W. H. Freeman & Co. (Référence de base sur le concept de NP-Complétude).
2. **Karp, R. M. (1972).** *Reducibility among combinatorial problems*. Complexity of computer computations (pp. 85-103). (Preuve fondatrice de la NP-complétude de ce type de problèmes logistiques).
3. **Savelsbergh, M. W. (1985).** *Local search in routing problems with time windows*. Annals of Operations Research, 4(1), 285-305. (Ouvrage de base sur la complexité spécifique induite par l'ajout des Time Windows au TSP).
4. **Desrochers, M., Lenstra, J. K., Savelsbergh, M. W., & Soumis, F. (1988).** *Vehicle routing with time windows: Optimization and approximation*. Vehicle routing: Methods and studies, 16(1), 65-84. (Techniques de résolutions pour notre problème précis).